# 🔄 CDC Snapshotting — Log Compaction with Trino & Iceberg

Compact an **append-only CDC log** into a materialized **current-state snapshot** using batch SQL — no Kafka or Flink required. Inspired by [Pinterest's DB Ingestion](https://medium.com/pinterest-engineering/next-generation-db-ingestion-at-pinterest-66844b7153b7).

```
Source DB ──CDC──▶ 🥉 Bronze (append-only ledger) ──MERGE──▶ 🥈 Silver (current-state snapshot)
                  Every I/U/D event logged             Dedup + MERGE INTO for latest state
```

**Prerequisites:** Run `setup.ipynb` first.

---
## ⚙️ Connect to Trino

In [1]:
from trino.dbapi import connect

conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="iceberg",
    schema="bronze",
)
cursor = conn.cursor()


def run_query(sql, display=True):
    """Execute a query and return results as a formatted table."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [
                max(len(str(c)), max(len(str(r[i])) for r in rows))
                for i, c in enumerate(columns)
            ]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []


print("✅ Connected to Trino")

✅ Connected to Trino


---
## 🥉 Bronze Layer — Append-Only CDC Ledger

Every row-level change (`I`nsert / `U`pdate / `D`elete) from the source DB is appended as an immutable event. This table is **never updated** — every change is a new row.

In [2]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.bronze.cdc_users_log (
    event_id   VARCHAR,
    op         VARCHAR,
    user_id    BIGINT,
    name       VARCHAR,
    email      VARCHAR,
    city       VARCHAR,
    event_ts   TIMESTAMP(6) WITH TIME ZONE
) WITH (format = 'PARQUET')
""")
print("✅ Table 'cdc_users_log' created in bronze layer")

✅ Table 'cdc_users_log' created in bronze layer


### 📥 Batch 1 — Initial Inserts

In [3]:
run_query("""
INSERT INTO iceberg.bronze.cdc_users_log VALUES
    ('evt001', 'I', 1, 'Alice',   'alice@old.com',    'Riyadh', TIMESTAMP '2026-02-24 08:00:00.000000 UTC'),
    ('evt002', 'I', 2, 'Bob',     'bob@email.com',    'Jeddah', TIMESTAMP '2026-02-24 08:05:00.000000 UTC'),
    ('evt003', 'I', 3, 'Charlie', 'charlie@email.com', 'Dammam', TIMESTAMP '2026-02-24 08:10:00.000000 UTC')
""")
print("✅ 3 CDC events ingested (Batch 1 — initial inserts)")

rows
----
3   
✅ 3 CDC events ingested (Batch 1 — initial inserts)


---
## 🥈 Silver Layer — Current State Snapshot

Answers: _"What does the table look like **right now**?"_

In [4]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.silver.users_snapshot (
    user_id    BIGINT,
    name       VARCHAR,
    email      VARCHAR,
    city       VARCHAR,
    updated_at TIMESTAMP(6) WITH TIME ZONE
) WITH (format = 'PARQUET')
""")
print("✅ Table 'users_snapshot' created in silver layer")

✅ Table 'users_snapshot' created in silver layer


### 🔄 The MERGE Pipeline

A CTE deduplicates via `ROW_NUMBER()` to get the latest event per `user_id`, then `MERGE INTO` applies inserts, updates, and deletes in one statement.

```
CDC Log (N events)                  Dedup CTE (1 per user)           Snapshot
┌─────────────────────┐            ┌─────────────────────┐          ┌─────────────────────┐
│ evt001 I Alice  RUH │            │                     │          │                     │
│ evt004 U Alice  DXB │ ──dedup──▶ │ latest Alice event  │ ─merge─▶ │ Alice  (upsert)     │
│ evt005 U Alice  DXB │            │ latest Bob event    │          │ Bob    (upsert)     │
│ ...                 │            │ latest Charlie (D)  │          │ (Charlie deleted)   │
└─────────────────────┘            └─────────────────────┘          └─────────────────────┘
```

In [5]:
MERGE_SQL = """
MERGE INTO iceberg.silver.users_snapshot AS tgt
USING (
    WITH deduped AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY event_ts DESC
            ) AS rn
        FROM iceberg.bronze.cdc_users_log
    )
    SELECT op, user_id, name, email, city, event_ts
    FROM deduped
    WHERE rn = 1
) AS src
ON tgt.user_id = src.user_id
WHEN MATCHED AND src.op = 'D' THEN
    DELETE
WHEN MATCHED AND src.op != 'D' THEN
    UPDATE SET
        name       = src.name,
        email      = src.email,
        city       = src.city,
        updated_at = src.event_ts
WHEN NOT MATCHED AND src.op != 'D' THEN
    INSERT (user_id, name, email, city, updated_at)
    VALUES (src.user_id, src.name, src.email, src.city, src.event_ts)
"""


def merge_cdc_to_snapshot():
    """Run the dedup + MERGE pipeline."""
    run_query(MERGE_SQL)
    print("✅ MERGE completed — snapshot updated!")


print("✅ MERGE pipeline defined")

✅ MERGE pipeline defined


### 🔄 Initial Sync — Batch 1

In [6]:
merge_cdc_to_snapshot()

rows
----
3   
✅ MERGE completed — snapshot updated!


In [7]:
print("📋 Current snapshot (baseline):")
print()
run_query("SELECT * FROM iceberg.silver.users_snapshot ORDER BY user_id");

📋 Current snapshot (baseline):

user_id | name    | email             | city   | updated_at               
--------+---------+-------------------+--------+--------------------------
1       | Alice   | alice@old.com     | Riyadh | 2026-02-24 08:00:00+00:00
2       | Bob     | bob@email.com     | Jeddah | 2026-02-24 08:05:00+00:00
3       | Charlie | charlie@email.com | Dammam | 2026-02-24 08:10:00+00:00


---
### 📥 Batch 2 — Updates & Deletes

Alice has **2 updates** in this batch plus her original insert — **3 total events** across the log. This is the key challenge for MERGE.

In [8]:
run_query("""
INSERT INTO iceberg.bronze.cdc_users_log VALUES
    ('evt004', 'U', 1, 'Alice',   'alice@old.com',    'Dubai',  TIMESTAMP '2026-02-24 09:00:00.000000 UTC'),
    ('evt005', 'U', 1, 'Alice',   'alice@new.com',    'Dubai',  TIMESTAMP '2026-02-24 10:30:00.000000 UTC'),
    ('evt006', 'U', 2, 'Bob',     'bob@email.com',    'Makkah', TIMESTAMP '2026-02-24 11:00:00.000000 UTC'),
    ('evt007', 'D', 3, 'Charlie', 'charlie@email.com', 'Dammam', TIMESTAMP '2026-02-24 12:00:00.000000 UTC')
""")
print("✅ 4 CDC events ingested (Batch 2)")

rows
----
4   
✅ 4 CDC events ingested (Batch 2)


### ❌ Why Naïve MERGE Fails

Trino's `MERGE` requires each target row to match **at most one** source row. Alice matches 3 — this errors out:

In [9]:
try:
    run_query("""
    MERGE INTO iceberg.silver.users_snapshot AS tgt
    USING iceberg.bronze.cdc_users_log AS src
    ON tgt.user_id = src.user_id
    WHEN MATCHED THEN
        UPDATE SET name = src.name, email = src.email, city = src.city, updated_at = src.event_ts
    WHEN NOT MATCHED THEN
        INSERT (user_id, name, email, city, updated_at)
        VALUES (src.user_id, src.name, src.email, src.city, src.event_ts)
    """)
except Exception as e:
    print(f"❌ MERGE FAILED (expected): {e}")
    print("💡 Fix: deduplicate with ROW_NUMBER() before MERGE — see merge_cdc_to_snapshot()")

❌ MERGE FAILED (expected): TrinoUserError(type=USER_ERROR, name=MERGE_TARGET_ROW_MULTIPLE_MATCHES, message="One MERGE target table row matched more than one source row", query_id=20260304_233835_00018_vqx8x)
💡 Fix: deduplicate with ROW_NUMBER() before MERGE — see merge_cdc_to_snapshot()


### ✅ Correct MERGE with Dedup

In [10]:
merge_cdc_to_snapshot()

rows
----
3   
✅ MERGE completed — snapshot updated!


In [11]:
run_query("SELECT * FROM iceberg.silver.users_snapshot ORDER BY user_id");

user_id | name  | email         | city   | updated_at               
--------+-------+---------------+--------+--------------------------
1       | Alice | alice@new.com | Dubai  | 2026-02-24 10:30:00+00:00
2       | Bob   | bob@email.com | Makkah | 2026-02-24 11:00:00+00:00


---
### 📥 Batch 3 — New User + More Updates

In [12]:
run_query("""
INSERT INTO iceberg.bronze.cdc_users_log VALUES
    ('evt008', 'I', 4, 'Dana',  'dana@email.com',  'Cairo',  TIMESTAMP '2026-02-24 14:00:00.000000 UTC'),
    ('evt009', 'U', 1, 'Alice', 'alice@new.com',   'London', TIMESTAMP '2026-02-24 15:00:00.000000 UTC'),
    ('evt010', 'U', 2, 'Bob',   'bob@newmail.com', 'Makkah', TIMESTAMP '2026-02-24 16:00:00.000000 UTC')
""")
print("✅ 3 CDC events ingested (Batch 3)")

rows
----
3   
✅ 3 CDC events ingested (Batch 3)


In [13]:
merge_cdc_to_snapshot()

rows
----
3   
✅ MERGE completed — snapshot updated!


In [14]:
print("📋 Full CDC log:")
print()
run_query("SELECT event_id, op, user_id, name, email, city, event_ts FROM iceberg.bronze.cdc_users_log ORDER BY event_ts");
print()
print("📋 Final snapshot:")
print()
run_query("SELECT * FROM iceberg.silver.users_snapshot ORDER BY user_id");

📋 Full CDC log:

event_id | op | user_id | name    | email             | city   | event_ts                 
---------+----+---------+---------+-------------------+--------+--------------------------
evt001   | I  | 1       | Alice   | alice@old.com     | Riyadh | 2026-02-24 08:00:00+00:00
evt002   | I  | 2       | Bob     | bob@email.com     | Jeddah | 2026-02-24 08:05:00+00:00
evt003   | I  | 3       | Charlie | charlie@email.com | Dammam | 2026-02-24 08:10:00+00:00
evt004   | U  | 1       | Alice   | alice@old.com     | Dubai  | 2026-02-24 09:00:00+00:00
evt005   | U  | 1       | Alice   | alice@new.com     | Dubai  | 2026-02-24 10:30:00+00:00
evt006   | U  | 2       | Bob     | bob@email.com     | Makkah | 2026-02-24 11:00:00+00:00
evt007   | D  | 3       | Charlie | charlie@email.com | Dammam | 2026-02-24 12:00:00+00:00
evt008   | I  | 4       | Dana    | dana@email.com    | Cairo  | 2026-02-24 14:00:00+00:00
evt009   | U  | 1       | Alice   | alice@new.com     | London | 2026-02-

---
## 🧊 Iceberg Snapshots

Every MERGE creates an immutable Iceberg snapshot — full audit trail of the table's evolution.

In [15]:
print("📸 Snapshot history:")
print()
run_query("""
SELECT committed_at, snapshot_id, parent_id, operation
FROM iceberg.silver."users_snapshot$snapshots"
ORDER BY committed_at
""");

📸 Snapshot history:

committed_at                     | snapshot_id         | parent_id           | operation
---------------------------------+---------------------+---------------------+----------
2026-03-04 23:37:08.333000+00:00 | 751454985152156940  | None                | append   
2026-03-04 23:38:31.019000+00:00 | 7186456318122520262 | 751454985152156940  | overwrite
2026-03-04 23:40:28.415000+00:00 | 4140799419842379923 | 7186456318122520262 | overwrite
2026-03-04 23:40:40.090000+00:00 | 522883551846698023  | 4140799419842379923 | overwrite


---
## 📊 Summary

| Concept | Implementation |
|---------|---------------|
| **Bronze** | Append-only CDC ledger (`I`/`U`/`D` events) |
| **Dedup** | `ROW_NUMBER() OVER (PARTITION BY pk ORDER BY ts DESC)` |
| **MERGE** | Single statement handles insert, update, and delete |
| **Silver** | Materialized current-state snapshot |

**Production considerations:** incremental high-water mark processing, partitioning by date, and scheduled maintenance via Airflow or similar. See `maintenance.ipynb` for Iceberg table maintenance operations.

---
## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to drop all tables:
# run_query("DROP TABLE IF EXISTS iceberg.silver.users_snapshot")
# run_query("DROP TABLE IF EXISTS iceberg.bronze.cdc_users_log")
# print("🗑️ All CDC tables dropped")